# Portfolio peak and evening-share report — corrected version

Each **Fix n** note refers to `mock_14_solution.md`.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

hh = pd.read_csv("../../data/meter_halfhourly_2023.csv.gz")
meters = pd.read_csv("../../data/meters.csv")
print(hh.shape, "| exact duplicate rows:", hh.duplicated().sum(),
      "| duplicate keys:", hh.duplicated(["meter_id", "settlement_date", "settlement_period"]).sum())

(349439, 4) | exact duplicate rows: 40 | duplicate keys: 40


**Fix 3** — drop the 40 duplicated rows before anything is summed.
**Fix 1 / 2** — the settlement date is a *local* (Europe/London) date. Localise midnight,
convert to UTC, then add the period offset. Periods 47–50 on the October clock-change day
and the 46-period March day then land on the right instants automatically.

In [2]:
hh = hh.drop_duplicates().copy()
local_midnight = pd.to_datetime(hh["settlement_date"]).dt.tz_localize("Europe/London")
hh["utc"] = local_midnight.dt.tz_convert("UTC") + pd.to_timedelta((hh["settlement_period"] - 1) * 30, unit="min")
hh = hh.sort_values(["meter_id", "utc"]).reset_index(drop=True)

periods_per_day = hh.groupby("settlement_date")["settlement_period"].max()
print(periods_per_day.value_counts().sort_index().to_dict())
print("DST days:", periods_per_day[periods_per_day != 48].to_dict())
print("all UTC timestamps unique per meter:", not hh.duplicated(["meter_id", "utc"]).any())

{46: 1, 48: 363, 50: 1}
DST days: {'2023-03-26': 46, '2023-10-29': 50}


all UTC timestamps unique per meter: True


**Fix 5 / 6 / 7** — data-quality flags before aggregation: the unit fault (M100007 in Wh
for September), the stuck week (M100003 in June) and the missing fortnight (M100011 in April).

In [3]:
hh["month"] = hh["utc"].dt.tz_convert("Europe/London").dt.month
mm = hh.groupby(["meter_id", "month"])["kwh"].median().unstack()
unit_ratio = mm.div(mm.median(axis=1), axis=0)
bad = unit_ratio.stack()[unit_ratio.stack() > 100]
print("x1000 meter-months:", bad.round(0).to_dict())
for mid, mon in bad.index:
    sel = (hh["meter_id"] == mid) & (hh["month"] == mon)
    hh.loc[sel, "kwh"] /= 1000

daily = hh.groupby(["meter_id", "settlement_date"])["kwh"].agg(n="size", nunique="nunique")
stuck = daily[daily["nunique"] <= 1]
print("stuck meter-days:", len(stuck), "for", stuck.index.get_level_values(0).unique().tolist())
hh = hh.set_index(["meter_id", "settlement_date"]).drop(stuck.index).reset_index()

expected = hh.groupby("meter_id")["utc"].agg(["min", "max"])
coverage = hh.groupby("meter_id").size() / 17520
print("coverage < 99%:", coverage[coverage < 0.99].round(3).to_dict())

x1000 meter-months: {('M100007', 9): 744.0}
stuck meter-days: 7 for ['M100003']


coverage < 99%: {'M100003': 0.98, 'M100011': 0.961}


**Fix 4** — kWh is energy: an hour's energy is the *sum* of its two half-hours, not the mean.
`resample("h")` on the UTC index; hours with only one half-hour present are marked incomplete.

In [4]:
g = hh.set_index("utc").groupby("meter_id")["kwh"]
hourly = pd.concat([g.resample("h").sum(min_count=2).rename("kwh"),
                    g.resample("h").size().rename("n_hh")], axis=1).reset_index()
hourly["local"] = hourly["utc"].dt.tz_convert("Europe/London")
hourly["hour_local"] = hourly["local"].dt.hour
print("incomplete hours:", (hourly["n_hh"] < 2).sum(), "| total hours:", len(hourly))

incomplete hours: 833 | total hours: 175200


**Fix 5 (result)** — the largest customers are the SMEs once the Wh month is corrected.

In [5]:
annual = hourly.groupby("meter_id")["kwh"].sum().rename("annual_kwh").sort_values(ascending=False)
annual.head(5).round(0)

meter_id
M100015    38247.0
M100010    22916.0
M100000    22550.0
M100016     5497.0
M100011     4542.0
Name: annual_kwh, dtype: float64

**Fix 10 / 11** — a half-hourly kWh reading is average power × 0.5 h, so kW = kWh × 2 (or
hourly kWh × 1). The portfolio requirement is the peak of the *coincident* sum, not the
sum of individual peaks.

In [6]:
hh["kw"] = hh["kwh"] * 2
peak_kw = hh.groupby("meter_id")["kw"].max().rename("peak_kw")
coincident = hh.groupby("utc")["kw"].sum()
n_at = hh.groupby("utc").size()
coincident = coincident[n_at == n_at.max()]           # only instants where all meters report
print(f"sum of individual peaks : {peak_kw.sum():8.1f} kW")
print(f"coincident portfolio peak: {coincident.max():8.1f} kW at {coincident.idxmax().tz_convert('Europe/London')}")
print(f"diversity factor         : {coincident.max() / peak_kw.sum():.2f}")

sum of individual peaks :    103.5 kW
coincident portfolio peak:     55.9 kW at 2023-11-24 17:00:00+00:00
diversity factor         : 0.54


**Fix 8** — the 17:00–20:00 window must be applied to *local* hours (and `between(17, 20)` is
four hours, 17:00–20:59; the benchmark window is three: 17, 18, 19).

In [7]:
ok = hourly[hourly["n_hh"] == 2]
evening = ok[ok["hour_local"].isin([17, 18, 19])]
evening_share = evening["kwh"].sum() / ok["kwh"].sum()
res_ids = meters.loc[meters["customer_type"] == "residential", "meter_id"]
ok_res = ok[ok["meter_id"].isin(res_ids)]
evening_share_res = ok_res.loc[ok_res["hour_local"].isin([17, 18, 19]), "kwh"].sum() / ok_res["kwh"].sum()
print(f"evening share (all meters)  : {evening_share:.1%}")
print(f"evening share (residential) : {evening_share_res:.1%}   (benchmark 22% is for the residential book)")

evening share (all meters)  : 15.9%
evening share (residential) : 23.0%   (benchmark 22% is for the residential book)


**Fix 9** — the reconciliation join lost meters because ids were upper-cased on one side only.
Normalise both keys, use `how="left"` with `indicator=True`, and annualise M100011 for its
missing fortnight before calling anything an "efficiency gain".

In [8]:
m = meters.copy()
m["meter_id"] = m["meter_id"].str.upper()
recon = (annual.reset_index().assign(meter_id=lambda d: d["meter_id"].str.upper())
           .merge(m[["meter_id", "customer_type", "annual_kwh_estimate"]], on="meter_id", how="left", indicator=True))
print(recon["_merge"].value_counts().to_dict())
recon = recon.merge(coverage.rename("coverage").reset_index(), on="meter_id")
recon["annual_kwh_scaled"] = recon["annual_kwh"] / recon["coverage"]
recon["ratio"] = recon["annual_kwh_scaled"] / recon["annual_kwh_estimate"]
print("meters reconciled:", len(recon), "| median ratio:", round(recon["ratio"].median(), 3))
recon.loc[recon["meter_id"].isin(["M100011", "M100007", "M100003"]),
          ["meter_id", "annual_kwh", "coverage", "annual_kwh_scaled", "annual_kwh_estimate", "ratio"]].round(3)

{'both': 20, 'left_only': 0, 'right_only': 0}
meters reconciled: 20 | median ratio: 1.044


,meter_id,annual_kwh,coverage,annual_kwh_scaled,annual_kwh_estimate,ratio
4,M100011,4542.267,0.961,4725.965,4511.0,1.048
6,M100007,4156.102,0.999,4159.426,3982.0,1.045
16,M100003,2651.535,0.980,2706.373,2575.0,1.051


## Honest results

In [9]:
pd.Series({
    "coincident_portfolio_peak_kw": round(coincident.max(), 1),
    "sum_of_individual_peaks_kw": round(peak_kw.sum(), 1),
    "evening_share_all": round(evening_share, 4),
    "evening_share_residential": round(evening_share_res, 4),
    "largest_meter": annual.index[0],
    "largest_meter_kwh": round(annual.iloc[0]),
    "meters_reconciled": len(recon),
    "median_actual_vs_estimate": round(recon["ratio"].median(), 3),
    "M100011_ratio_after_annualising": round(recon.set_index("meter_id").loc["M100011", "ratio"], 3),
})

coincident_portfolio_peak_kw          55.9
sum_of_individual_peaks_kw           103.5
evening_share_all                   0.1592
evening_share_residential           0.2295
largest_meter                      M100015
largest_meter_kwh                    38247
meters_reconciled                       20
median_actual_vs_estimate            1.044
M100011_ratio_after_annualising      1.048
dtype: object